In [1]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Define the area (England and Wales)
# boundary = ox.geocode_to_gdf("England, United Kingdom").to_crs(4326)
boundary = ox.geocode_to_gdf("United Kingdom").to_crs(4326)
boundary_geom = boundary.geometry.iloc[0]

# Parameters
num_splits = 10  # can increase to 6–8 for more parallelism
max_workers = 16
amenities = [
    "restaurant", "cafe", "fast_food", "bank",
    "pharmacy", "bar", "pub", "post_office",
    "marketplace", "nightclub", "bureau_de_change", "food_court"
]

# Split the boundary polygon into grid tiles
def split_area(geometry, num_splits=4):
    minx, miny, maxx, maxy = geometry.bounds
    dx = (maxx - minx) / num_splits
    dy = (maxy - miny) / num_splits
    tiles = []
    for i in range(num_splits):
        for j in range(num_splits):
            tile = box(minx + i * dx, miny + j * dy, minx + (i + 1) * dx, miny + (j + 1) * dy)
            if tile.intersects(geometry):
                tiles.append(tile.intersection(geometry))
    return tiles

tiles = split_area(boundary_geom, num_splits=num_splits)

# Query function for a single amenity in a tile
def query_amenity(tile_geom, amenity):
    try:
        tags = {"amenity": amenity}
        gdf = ox.features_from_polygon(tile_geom, tags)
        if gdf.empty:
            return None
        gdf = gdf.reset_index()
        gdf = gdf.rename(columns={gdf.columns[0]: "element_type", gdf.columns[1]: "osmid"})
        gdf["geometry"] = gdf.geometry.centroid
        gdf["value"] = amenity
        gdf["key"] = "amenity"
        gdf = gdf.drop_duplicates(subset="osmid")
        return gdf[["osmid", "name", "value", "key", "geometry"]]
    except Exception as e:
        return None

# Parallel query across tiles and amenities
def fetch_amenities_parallel(tiles, amenities):
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(query_amenity, tile, amenity)
                   for tile in tiles for amenity in amenities]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching amenities"):
            try:
                result = future.result()
                if result is not None and not result.empty:
                    results.append(result)
            except Exception as e:
                print(f"Error fetching amenity: {e}")
    return gpd.GeoDataFrame(pd.concat(results, ignore_index=True), crs="EPSG:4326") if results else gpd.GeoDataFrame(columns=["osmid", "name", "value", "key", "geometry"], crs="EPSG:4326")

# Query shops similarly
def query_shops(tile_geom):
    try:
        tags = {"shop": True}
        gdf = ox.features_from_polygon(tile_geom, tags)
        if gdf.empty:
            return None
        gdf = gdf.reset_index()
        gdf = gdf.rename(columns={gdf.columns[0]: "element_type", gdf.columns[1]: "osmid"})
        gdf["geometry"] = gdf.geometry.centroid
        gdf["value"] = gdf["shop"]
        gdf["key"] = "shop"
        gdf = gdf.drop_duplicates(subset="osmid")
        return gdf[["osmid", "name", "value", "key", "geometry"]]
    except Exception as e:
        return None

def fetch_shops_parallel(tiles):
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(query_shops, tile) for tile in tiles]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching shops"):
            try:
                result = future.result()
                if result is not None and not result.empty:
                    results.append(result)
            except Exception as e:
                print(f"Error fetching shop: {e}")
    return gpd.GeoDataFrame(pd.concat(results, ignore_index=True), crs="EPSG:4326") if results else gpd.GeoDataFrame(columns=["osmid", "name", "value", "key", "geometry"], crs="EPSG:4326")

# Run both fetches
amenity_pts = fetch_amenities_parallel(tiles, amenities)
shop_pts = fetch_shops_parallel(tiles)

# Combine and export or analyze
all_pts = pd.concat([amenity_pts, shop_pts], ignore_index=True)
all_pts = gpd.GeoDataFrame(all_pts, crs="EPSG:4326")
print(f"Total points: {len(all_pts)}")


Fetching shops: 100%|██████████| 53/53 [06:06<00:00,  6.91s/it]

Total points: 470748


In [3]:
all_pts

#save to parquet
output_file = "amenities_and_shops_sheffield.parquet"
all_pts.to_parquet(output_file, index=False)

In [ ]:


# Get the UK boundary
uk_boundary = ox.geocode_to_gdf("United Kingdom").geometry[0]
# Query retail landuse data in parallel with progress tracking
retail_landuse = query_with_progress_concurrent(uk_boundary, num_splits=10, max_workers=16)

# Clip the data to the UK boundary and remove duplicates
retail_landuse = (
    gpd.clip(retail_landuse, uk_boundary)
    [['geometry', 'landuse', 'name']]
    .drop_duplicates()
)
#save to file (geoparquet)
retail_landuse.to_parquet("data/input_files/retail_landuse.parquet")

Querying Sub-areas: 100%|██████████| 100/100 [00:22<00:00,  4.50it/s]


In [ ]:


# Function to convert polygon or multipolygon to H3 indices
def get_h3_indices(geometry, resolution=13):
    # Check if the geometry is a Polygon or MultiPolygon
    if geometry.geom_type == 'Polygon':
        coords = [(lat, lng) for lng, lat in geometry.exterior.coords]
    elif geometry.geom_type == 'MultiPolygon':
        coords = []
        # Use geoms to iterate over individual polygons in the MultiPolygon
        for poly in geometry.geoms:
            coords.extend([(lat, lng) for lng, lat in poly.exterior.coords])
    else:
        raise ValueError(f"Geometry type {geometry.geom_type} not supported")

    # Create a LatLngPoly object
    poly = h3.LatLngPoly(coords)
    # Convert the polygon to H3 cells at the specified resolution
    return h3.h3shape_to_cells(poly, res=resolution)

# # Apply the function to the GeoDataFrame to get H3 indices at resolution 13
retail_landuse['h3_indices_res13'] = retail_landuse['geometry'].apply(get_h3_indices, resolution=13)
retail_landuse['h3_indices_res11'] = retail_landuse['geometry'].apply(get_h3_indices, resolution=11)

# Now apply the function to convert the H3 indices to resolution 11
#drop rows with empty h3_indices_res13, these are a handful of very small polygons
retail_landuse = retail_landuse[retail_landuse['h3_indices_res13'].apply(len) > 0]
retail_landuse['h3_indices_res11_parents'] = retail_landuse['h3_indices_res13'].apply(lambda h3_cells: [h3.cell_to_parent(cell, 11) for cell in h3_cells])
#explode h3_indices_res11_parents and rename to h3_address
retail_landuse_h3 = retail_landuse.explode('h3_indices_res11_parents').rename(columns={'h3_indices_res11_parents': 'h3_11'}).drop(columns=['h3_indices_res13', 'h3_indices_res11',"name","geometry"])
#drop repeats

retail_landuse_h3 = retail_landuse_h3.drop_duplicates()

#save to file (parquet)
retail_landuse_h3.to_parquet("data/input_files/retail_landuse_h3.parquet")